# 2. Base finale : jointures et variables

Les tables propres de `data/clean/` sont réunies dans `data/clean/base_finale/`.

In [1]:
import os
import sys

import geopandas as gpd
import pandas as pd
import yaml

# on se place à la racine du projet
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.append(os.getcwd())

# paramètres communs du projet
with open("config.yaml", encoding="utf-8") as f:
    config = yaml.safe_load(f)


## 2.1 Les deux bases SNCF

### 1. Clé de jointure

Clé de jointure : code_uic

L'unicité de la clé dans la table gares-de-voyageurs.parquet a été vérifiée dans la fonction *clean_gares()* (étape 5)

On vérifie le nombre de code_uic par ligne dans la table *gares-de-voyageurs*

In [2]:
from src.nettoyage import clean_frequentation
from src.nettoyage import clean_gares
from src.jointure import verif_code_uic

base = [clean_gares(),clean_frequentation()]

verif_code_uic(base)


       position_geographique code_commune  code_uic
0      49.6852237, 1.7743058        60001  87313759
1     47.5546432, -1.5244159        44001  87481614
2          50.10221, 1.82449        80001  87317362
3        48.725468, 2.419151        94001  87545269
4      48.9551835, 2.0919031        78551  87386052
...                      ...          ...       ...
2787  48.4702462, -2.6524629        22389  87473116
2788  43.9781851, -0.7361528        40333  87671487
2789   44.9106891, 2.3644474        15267  87645101
2790     49.622035, 0.750115        76758  87413385
2791   47.6943143, 7.2945376        68384  87182196

[2792 rows x 3 columns]
11
                    nom_gare  code_uic code_postal segmentation_drg  annee  \
0                   Abbaretz  87481614       44170                C   2023   
1      Achères Grand Cormier  87386052       78100                B   2023   
2            Achiet-le-Grand  87342048       62121                C   2023   
3                       Agay  877575

Dans la base sgares-de-voyageurs*, 11 gares ont plusieurs *code_uic*. Ces codes multiples n'apparaissent pas dans la base *frequentation-gares* ce qui a posé un problème lors des premières tentatives de jointures (correction à l'étape 2 de la fonction *join_sncf())*

### 2. Type de jointure

Dans un premier temps, nous réalisé la jointure avec comme l'arugment "outer" pour la fonction merge *join_gares_audit()* afin de conserver toutes les gares et de comprendre pourquoi certaines n'apparaissent dans une base ou dans l'autre.

In [3]:
# On récupère le fichier fusionné en appelant la fonction join_sncf_audit depuis src.jointure
from src.jointure import join_sncf_audit

sncf_audit = join_sncf_audit()

# On affiche les dimensions du fichier, les premières lignes et les types des variables
print(sncf_audit.shape)
display(sncf_audit.info())
display(sncf_audit.head())

(21092, 9)
<class 'pandas.DataFrame'>
RangeIndex: 21092 entries, 0 to 21091
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype   
---  ------                 --------------  -----   
 0   position_geographique  19503 non-null  str     
 1   code_commune           19503 non-null  str     
 2   code_uic               21092 non-null  str     
 3   nom_gare               21070 non-null  str     
 4   code_postal            21070 non-null  str     
 5   segmentation_drg       20601 non-null  str     
 6   annee                  21070 non-null  float64 
 7   frequentation          21070 non-null  float64 
 8   _merge                 21092 non-null  category
dtypes: category(1), float64(2), str(6)
memory usage: 2.3 MB


None

,position_geographique,code_commune,code_uic,nom_gare,code_postal,segmentation_drg,annee,frequentation,_merge
0,"49.003652, 2.570892",93073,87001479,NaN,NaN,NaN,NaN,NaN,left_only
1,"47.3388167, 0.6528244",37122,87009696,La Douzillère,37300,C,2023.0,6573.0,both
2,"47.3388167, 0.6528244",37122,87009696,La Douzillère,37300,C,2022.0,9903.0,both
3,"47.3388167, 0.6528244",37122,87009696,La Douzillère,37300,C,2021.0,9147.0,both
4,"47.3388167, 0.6528244",37122,87009696,La Douzillère,37300,C,2020.0,7664.0,both


In [4]:
# Lignes présentes uniquement dans gares
gares_non_appariees = sncf_audit[sncf_audit["_merge"] == "left_only"]

display(gares_non_appariees)
print(len(gares_non_appariees))

,position_geographique,code_commune,code_uic,nom_gare,code_postal,segmentation_drg,annee,frequentation,_merge
0,"49.003652, 2.570892",93073,87001479,NaN,NaN,NaN,NaN,NaN,left_only
3557,"48.880185, 2.355151",75110,87271023,NaN,NaN,NaN,NaN,NaN,left_only
3558,"48.880185, 2.355151",75110,87271031,NaN,NaN,NaN,NaN,NaN,left_only
7143,"48.8807466, 2.0790676",78367,87382812,NaN,NaN,NaN,NaN,NaN,left_only
7186,"48.897377, 2.224397",92050,87386011,NaN,NaN,NaN,NaN,NaN,left_only
7334,"48.841172, 2.320514",75115,87391011,NaN,NaN,NaN,NaN,NaN,left_only
7335,"48.841172, 2.320514",75115,87391102,NaN,NaN,NaN,NaN,NaN,left_only
7728,"48.8415189, 2.0618388",78455,87393876,NaN,NaN,NaN,NaN,NaN,left_only
10809,"48.98032, 2.27176",95219,87534131,NaN,NaN,NaN,NaN,NaN,left_only
11223,"48.842285, 2.364891",75113,87547026,NaN,NaN,NaN,NaN,NaN,left_only


22


Au total 22 gares de *gares-de-voyageurs* n'apparaissent pas dans *frequentation-gares*, il n'y a donc pas d'observation de leur fréquentation, il semble donc pertinent de ne pas les garder dans notre base finale.

In [5]:
# Lignes présentes uniquement dans frequentation
frequentation_non_appariees = sncf_audit[sncf_audit["_merge"] == "right_only"]

display(frequentation_non_appariees)
print(len(frequentation_non_appariees))

,position_geographique,code_commune,code_uic,nom_gare,code_postal,segmentation_drg,annee,frequentation,_merge
701,NaN,NaN,87141531,Bainville-sur-Madon,54550,C,2023.0,7.0,right_only
702,NaN,NaN,87141531,Bainville-sur-Madon,54550,C,2022.0,84.0,right_only
703,NaN,NaN,87141531,Bainville-sur-Madon,54550,C,2021.0,6.0,right_only
704,NaN,NaN,87141531,Bainville-sur-Madon,54550,C,2020.0,2.0,right_only
705,NaN,NaN,87141531,Bainville-sur-Madon,54550,C,2019.0,60.0,right_only
...,...,...,...,...,...,...,...,...,...
21065,NaN,NaN,87784892,Bena Fanes,66760,C,2021.0,46.0,right_only
21066,NaN,NaN,87784892,Bena Fanes,66760,C,2020.0,23.0,right_only
21067,NaN,NaN,87784892,Bena Fanes,66760,C,2019.0,78.0,right_only
21068,NaN,NaN,87784892,Bena Fanes,66760,C,2018.0,33.0,right_only


1589


Nous constatons que certaines gares qui n'apparaissent pas dans dans *gares-de-voyageurs* ont fermées (ex : Bainvillle-sur-Madon) mais d'autres semblent encore ouvertes et accueillaient des voyageurs en 2025 (ex : Bena Fanes)

In [6]:
# Nombre de voyageurs dans les lignes non appariées
nb = frequentation_non_appariees["frequentation"].sum()
print(nb)

# Proportion des voyageurs dans les lignes non appariées sur le nombre total de voyageur
tot = clean_frequentation()["frequentation"].sum()
pct = (nb / tot) * 100
print(pct)

52335600.0
0.3071602450599038


Le nombre total de voyageurs passant dans les gares non appariées sur la période étudiée ne représente que 0.3% du nombre total de voyageurs sur cette période. Nous avons donc fait le choix de les négliger et d'utiliser inner join comme type de jointure dans notre nouvelle fonction *join_sncf_inner()*.

### 3. Résultat inner join

In [7]:
# On récupère le fichier fusionné en appelant la fonction join_sncf_inner depuis src.jointure
from src.jointure import join_sncf_inner

sncf_inner = join_sncf_inner()

# On affiche les premières lignes du fichier et ses dimensions
display(sncf_inner.head())
print(sncf_inner.shape)

,position_geographique,code_commune,code_uic,nom_gare,code_postal,segmentation_drg,annee,frequentation
0,"49.6852237, 1.7743058",60001,87313759,Abancourt,60220,C,2023,77974
1,"49.6852237, 1.7743058",60001,87313759,Abancourt,60220,C,2022,71517
2,"49.6852237, 1.7743058",60001,87313759,Abancourt,60220,C,2021,51811
3,"49.6852237, 1.7743058",60001,87313759,Abancourt,60220,C,2020,32396
4,"49.6852237, 1.7743058",60001,87313759,Abancourt,60220,C,2019,42685


(19481, 8)


Note : il y a 19481 lignes donc 2783 gares

### 4. Valeurs nulles



In [8]:
import pandas as pd

# Somme de Booleans qui teste les lignes où la fréquentation est à 0
nb_zero = (sncf_inner["frequentation"] == 0).sum()
print( "nombre de lignes à 0 :", nb_zero)

# Proportion des 0 sur le nombre total d'observations
pct_zero = (sncf_inner["frequentation"] == 0).mean() * 100
print("proportion de lignes à 0 :", pct_zero)

# Nombre de gare avec une ligne ou plus à  0
gares_zero = sncf_inner.groupby("code_uic")["frequentation"].apply(lambda x: (x == 0).any()).sum()
print( "nombre de gares avec au moins un 0 :", gares_zero)

# Proportion des gares avec une ligne ou plus à  0
pct_gares_zero = gares_zero / 2783 * 100
print("proportion de gares avec au moins une ligne à 0 :", pct_gares_zero)

# Afficher les lignes où des 0 apparaissent
zero = (sncf_inner[sncf_inner["frequentation"] == 0].groupby("code_uic").agg(nb_annees_zero=("annee", "count"),premiere_annee_zero=("annee", "min"),derniere_annee_zero=("annee", "max")).reset_index())
display(zero.loc[pd.col("derniere_annee_zero") == 2023])
#gares_zero.to_excel("gares_zero.xlsx", index=False)
#print(pd.crosstab(df_long["Segmentation DRG"],df_long["fréquentation"].eq(0),normalize="index") * 100)

nombre de lignes à 0 : 124
proportion de lignes à 0 : 0.6365176325650634
nombre de gares avec au moins un 0 : 39
proportion de gares avec au moins une ligne à 0 : 1.4013654329859864


,code_uic,nb_annees_zero,premiere_annee_zero,derniere_annee_zero
10,87472639,7,2017,2023
11,87534339,7,2017,2023
25,87725457,7,2017,2023
26,87725465,7,2017,2023


La proportion de lignes où la fréquentation vaut 0 nous semble suffisament faible pour la négliger.

Nous identifions 4 gares pour lesquelles la fréquentation est toujours égales à 0 sur notre périmètre, nous décidons donc de les supprimer dans notre fonction finale *join_sncf_final()*.

### 5. Résultat final

In [9]:
from src.jointure import join_sncf_final

sncf_final = join_sncf_final()
display(sncf_final.head())
print(sncf_final.shape)

,position_geographique,code_commune,code_uic,nom_gare,code_postal,segmentation_drg,annee,frequentation
0,"49.6852237, 1.7743058",60001,87313759,Abancourt,60220,C,2023,77974
1,"49.6852237, 1.7743058",60001,87313759,Abancourt,60220,C,2022,71517
2,"49.6852237, 1.7743058",60001,87313759,Abancourt,60220,C,2021,51811
3,"49.6852237, 1.7743058",60001,87313759,Abancourt,60220,C,2020,32396
4,"49.6852237, 1.7743058",60001,87313759,Abancourt,60220,C,2019,42685


(19453, 8)


In [10]:
# Nombre de lignes à 0
nb_zero = (sncf_final["frequentation"] == 0).sum()
print( "nombre de lignes à 0 :", nb_zero)

# Proportion de lignes à  0
pct_zero = (sncf_final["frequentation"] == 0).mean() * 100
print("proportion de lignes à 0 :", pct_zero)

# Nombre de gare avec une ligne ou plus à  0
gares_zero = sncf_final.groupby("code_uic")["frequentation"].apply(lambda x: (x == 0).any()).sum()
print( "nombre de gares avec au moins un 0 :", gares_zero)

# Proportion des gares avec une ligne ou plus à  0
pct_gares_zero = gares_zero /  (19453/7) * 100
print("proportion de gares avec au moins une ligne à 0 :", pct_gares_zero)

nombre de lignes à 0 : 96
proportion de lignes à 0 : 0.49349714696961905
nombre de gares avec au moins un 0 : 35
proportion de gares avec au moins une ligne à 0 : 1.2594458438287155


## 2.2 La table des communes

Une ligne par commune : contours, population, navettes, parc automobile, gares et voyageurs, distance à la gare.

In [11]:
from src.jointure import build_communes, build_distance_gare, build_gares, build_gares_annees, build_jointure, build_voyageurs_communes

# les tables propres de chaque base
contours = gpd.read_parquet("data/clean/contours_communes.parquet")
population = pd.read_parquet("data/clean/population.parquet")
navettes = pd.read_parquet("data/clean/navettes.parquet")
parc = pd.read_parquet("data/clean/parc_auto.parquet")
# table SNCF : une ligne par gare et par année
sncf = pd.read_parquet("data/clean/sncf-merged.parquet")

FileNotFoundError: data/clean/parc_auto.parquet

In [ ]:
gares = build_gares(sncf, contours)
voyageurs = build_voyageurs_communes(sncf, gares, config["annee_debut"], config["annee_fin"])
distance = build_distance_gare(contours, gares, config["crs_calcul"])
gares_annees = build_gares_annees(sncf, gares)

# la grande table : une ligne par commune
communes = build_communes(contours, population, navettes, parc, voyageurs)
communes = build_jointure(communes, distance, "distance à la gare")
print(communes.shape)
print("communes sans distance (Corse) :", communes["dist_gare_km"].isna().sum())
print(gares_annees.groupby("annee")["voyageurs"].sum())

methode
code SNCF                 2712
arrondissement              42
point dans le polygone      25
Name: count, dtype: int64
gares sans commune : 0
population : 34746 communes, 0 sans correspondance (0.00 %)
navettes : 34746 communes, 0 sans correspondance (0.00 %)
parc automobile : 34746 communes, 4 sans correspondance (0.01 %)
gares et voyageurs : 34746 communes, 32297 sans correspondance (92.95 %)
distance à la gare : 34746 communes, 0 sans correspondance (0.00 %)
(34746, 36)
communes sans distance (Corse) : 360
annee
2017    2723841659
2018    2675366476
2019    2724376670
2020    1423972270
2021    2068635456
2022    2590865445
2023    2779139516
Name: voyageurs, dtype: int64


In [ ]:
os.makedirs("data/clean/base_finale", exist_ok=True)
communes.to_parquet("data/clean/base_finale/base_finale_communes.parquet")
gares.to_parquet("data/clean/base_finale/base_finale_gares.parquet")
gares_annees.to_parquet("data/clean/base_finale/base_finale_gares_annees.parquet")

## 2.3 Les variables communales

Croissance, trajectoire, voyageurs, part de la voiture, densité, taille et profil de chaque commune.

In [ ]:
from src.variables import build_classe_taille, build_part, build_profil, build_tcam, build_trajectoire

communes = gpd.read_parquet("data/clean/base_finale/base_finale_communes.parquet")
nb_annees = config["annee_fin"] - config["annee_debut"]

In [ ]:
communes["tcam_pop_2017_2023"] = build_tcam(communes["pop_2017"], communes["pop_2023"], nb_annees)
communes["trajectoire"] = build_trajectoire(communes["tcam_pop_2017_2023"], config["seuil_stable_pct"])

print(communes["tcam_pop_2017_2023"].describe())
print(communes["trajectoire"].value_counts(dropna=False))

count    34740.000000
mean         0.077497
std          1.466141
min        -12.429141
25%         -0.717883
50%          0.036597
75%          0.828529
max         18.364677
Name: tcam_pop_2017_2023, dtype: float64
trajectoire
croissance    15447
déclin        14405
stable         4888
NaN               6
Name: count, dtype: int64


In [ ]:
# croissance des voyageurs des gares de la commune (NaN sans gare)
communes["tcam_voyageurs_2017_2023"] = build_tcam(communes["voyageurs_2017"], communes["voyageurs_2023"], nb_annees)

# voyageurs des gares de la commune par habitant (NaN sans gare)
communes["voyageurs_hab_2023"] = communes["voyageurs_2023"] / communes["pop_2023"].where(communes["pop_2023"] > 0)

# part des actifs allant travailler en voiture, NaN sous 1 000 actifs
for annee in [config["annee_debut"], config["annee_fin"]]:
    communes[f"part_voiture_{annee}"] = build_part(communes[f"actifs_voiture_{annee}"], communes[f"actifs_{annee}"], config["actifs_min"])

# part de la voiture dans toutes les communes, sans seuil d'actifs (pour les boîtes par taille de commune)
communes["part_voiture_toutes_2023"] = build_part(communes["actifs_voiture_2023"], communes["actifs_2023"], 1)

# voitures des particuliers pour 1 000 habitants
communes["voitures_1000_hab_2023"] = communes["voitures_2023"] / communes["pop_2023"].where(communes["pop_2023"] > 0) * 1000

In [ ]:
communes["densite"] = communes["pop_2023"] / communes["superficie_km2"]
communes["classe_taille"] = build_classe_taille(communes["pop_2023"], config["bornes_taille"], config["noms_taille"])
communes["profil"], communes["desservie"] = build_profil(communes["tcam_pop_2017_2023"], communes["dist_gare_km"], config["seuil_desservie_km"])

print(communes["classe_taille"].value_counts().sort_index())
print(communes["profil"].value_counts(dropna=False))

classe_taille
moins de 500       18252
500 à 2 000        11097
2 000 à 5 000       3240
5 000 à 10 000      1169
10 000 à 50 000      865
50 000 et plus       123
Name: count, dtype: int64
profil
croissance – desservie              10777
déclin ou stable – desservie         9240
déclin ou stable – non desservie     7752
croissance – non desservie           6611
NaN                                   366
Name: count, dtype: int64


In [ ]:
communes.to_parquet("data/clean/base_finale/base_finale_communes_variables.parquet")
print(communes.shape)

(34746, 48)


## 2.4 Les liaisons aériennes et leur CO2 (bonus)

In [ ]:
from src.variables import build_liaisons_co2

liaisons = pd.read_parquet("data/clean/liaisons_aeriennes.parquet")
facteurs = pd.read_parquet("data/clean/facteurs_emission.parquet")

liaisons_co2 = build_liaisons_co2(liaisons, facteurs, config["annee_debut"], config["annee_fin"])
print(liaisons_co2.shape)
print(liaisons_co2.filter(like="co2").sum().round())

(52, 17)
co2_avion_t_2017    1464021.0
co2_tgv_t_2017        17191.0
co2_avion_t_2023    1075604.0
co2_tgv_t_2023        12546.0
dtype: float64


In [ ]:
liaisons_co2.to_parquet("data/clean/base_finale/base_finale_liaisons_co2.parquet")